<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/gemini_colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

'AQ.Ab8RN6IrgyBUwfewaW1_C6S1edyXS5A6eyFJZpscdyWBg1IrEAB9yli_0KdwfuEzmDq6A9bEY_QxGhssy4'

# **大学のGOOGLEアカウントでは、APIキーの発行ができないようです。すて垢作るなど、別のアカウントで取得してください**

# Gemini API 入門 (Google Colab)

Google AI Studio の **無料枠** で Gemini API を叩く最小サンプルです。

## 事前準備
1. <https://aistudio.google.com/apikey> で API キーを発行（Google アカウントでログイン、カード不要）。
2. Colab 左側の **🔑（シークレット）** を開き、`GOOGLE_API_KEY` という名前でキーを登録。「ノートブックからのアクセス」をオンにする。

> ⚠️ 無料枠ではプロンプト・応答が Google の改善に使われる場合があります。個人情報や未公開データは入れないこと。


## 1. SDK のインストール

In [ ]:
!pip install -q -U google-genai

## 2. クライアントの準備
シークレットに登録した `GOOGLE_API_KEY` を読み込みます。未登録のときは手入力に切り替わります。

In [ ]:
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    import getpass
    API_KEY = getpass.getpass("Gemini API key: ")

client = genai.Client(api_key=API_KEY)

# 使うモデル。日次上限を増やしたいなら "gemini-2.5-flash-lite"、
# 難しい課題には "gemini-2.5-pro"（無料枠は上限が小さい）。
MODEL = "gemini-2.5-flash"
print("OK: client ready")

## 3. いちばん簡単な生成

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="フーリエ変換を高校生にもわかるように3文で説明して。",
)
print(response.text)

## 4. ストリーミング出力（生成され次第、逐次表示）

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="サンプリング定理のポイントを箇条書き4点で。",
):
    print(chunk.text, end="")

## 5. システム指示・パラメータの指定
`system_instruction` で役割を与え、`temperature` で出力のばらつきを調整します。

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="電気回路のインピーダンスとは？",
    config=types.GenerateContentConfig(
        system_instruction="あなたは大学の電子工学の教員です。簡潔に、数式は最小限で答えます。",
        temperature=0.3,
        max_output_tokens=300,
    ),
)
print(response.text)

## 6. マルチターン会話（文脈を保持）

In [ ]:
chat = client.chats.create(model=MODEL)
print(chat.send_message("私の名前は山田です。よろしく。").text)
print("---")
print(chat.send_message("私の名前は何でしたか？").text)

## 7.（任意）画像を使う = マルチモーダル
実行するとファイル選択ダイアログが出ます。画像を1枚選んでください。

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()          # 画像ファイルを選択
fname = next(iter(uploaded))
img = Image.open(fname)

response = client.models.generate_content(
    model=MODEL,
    contents=["この画像に何が写っているか日本語で説明して。", img],
)
print(response.text)

## 8. うまくいかないとき

- **429 (RESOURCE_EXHAUSTED)**: 無料枠のレート上限超過。`gemini-2.5-flash` は概ね 10 RPM / 250 RPD、`gemini-2.5-flash-lite` は 15 RPM / 1,000 RPD。少し待つ、`MODEL` を flash-lite に変える、指数バックオフ（1→2→4→8秒）で再試行。
- **PERMISSION_DENIED / 400**: キーが間違っているか未設定。シークレット名が `GOOGLE_API_KEY` か確認。
- **モデル名エラー**: 利用可能なモデルは `for m in client.models.list(): print(m.name)` で確認できる。
- **上限・地域差**: 無料枠は予告なく変わる。最新は <https://ai.google.dev/gemini-api/docs/rate-limits> を参照。

### 実習での注意
- キーは1人1つ（各自のアカウントで発行）。共有すると残高・上限をすぐ使い切る。
- ノートブックを共有・提出する前に、キーを直書きしていないか確認（シークレット運用なら安全）。
